# 3-hour SHARP Gray-Box training

This notebook is the horizon-specific training entry point. It uses the matched SHARP tensor and the provisional 3-hour labels only after the label and role gates pass. It never reuses 72-hour targets, thresholds, calibration maps, or checkpoints.

## Scope and claim boundary

The current notebook prepares and verifies the horizon-specific training matrix. Labels remain provisional until the event-source and role decisions are frozen. A successful run supports a horizon-specific SHARP experiment only; it does not establish AIA or fusion performance.

In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd

HORIZON = 3
ROOT = Path("data/processed/gray_box_aligned_v1")
LABEL_ROOT = Path(f"data/processed/gray_box_multihorizon_labels_{HORIZON}h_v1")
labels = pd.read_csv(LABEL_ROOT / f"candidate_labels_{HORIZON}h.csv.gz", low_memory=False)
index = pd.read_csv(ROOT / "input_index.csv.gz", low_memory=False)
sharp = np.load(ROOT / "sharp.npy", mmap_mode="r", allow_pickle=False)
assert len(index) == len(sharp) == 113433
assert labels.forecast_case_id.is_unique and index.forecast_case_id.is_unique
matched = labels.merge(index[["forecast_case_id","tensor_row","region_component_id","issue_utc"]], on="forecast_case_id", validate="one_to_one")
matched = matched[matched.tensor_row >= 0].copy()
print({"horizon": HORIZON, "matched_cases": len(matched), "tensor_shape": sharp.shape})

## Chronological roles

These boundaries mirror the 72-hour chronology for comparability. They must be reviewed and frozen separately for this horizon before model fitting.

In [ ]:
role_blocks = [
    ("train", "2010-01-01", "2014-01-01"),
    ("model_validation", "2014-01-01", "2014-07-01"),
    ("probability_calibration", "2014-07-01", "2015-01-01"),
    ("conformal_calibration", "2015-01-01", "2015-07-01"),
    ("policy_validation", "2015-07-01", "2020-01-01"),
    ("retrospective_cycle25", "2021-01-01", "2026-01-01"),
    ("supplementary_2026", "2026-01-01", "2027-01-01"),
]
matched["issue"] = pd.to_datetime(matched.issue_utc, utc=True)
matched["role"] = "outside_blocks"
for role, start, stop in role_blocks:
    mask = (matched.issue >= pd.Timestamp(start, tz="UTC")) & (matched.issue < pd.Timestamp(stop, tz="UTC"))
    matched.loc[mask, "role"] = role
known = matched.candidate_primary_label.notna()
matched.loc[~known, "role"] = "unknown_label_excluded"
print(matched.groupby("role", dropna=False).size().to_string())

## Leakage and support checks

In [ ]:
assert matched.groupby("forecast_case_id").size().max() == 1
assert matched.groupby("role").region_component_id.nunique().notna().all()
role_sets = {r:set(g.region_component_id) for r,g in matched[matched.role.isin([x[0] for x in role_blocks])].groupby("role")}
for a, sa in role_sets.items():
    for b, sb in role_sets.items():
        if a < b:
            print(a, b, "shared_region_components", len(sa & sb))
train = matched.role.eq("train") & matched.candidate_primary_label.notna()
assert train.any(), "No known training labels"
X = np.asarray(sharp[matched.loc[train, "tensor_row"].to_numpy()])
y = matched.loc[train, "candidate_primary_label"].astype(int).to_numpy()
print({{"training_cases": len(y), "positives": int(y.sum()), "prevalence": float(y.mean())}})

## Training handoff

After protocol approval, this matrix should be passed to a horizon-specific three-seed GRU runner. The runner must record the label-file hash, role boundaries, training indices, checkpoint hashes, and device. Do not report model results from this preparation notebook.